# Machine blindsight: Kaggle runner

Settings: **Accelerator = GPU T4 x2**, **Internet = On**.

Runs the whole experiment for three models. Results land in `/kaggle/working/machine-blindsight/results/` and are zipped at the end so you can download them.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /kaggle/working
!rm -rf machine-blindsight
!git clone --depth 1 https://github.com/pranavsaikosaraju/machine-blindsight.git
%cd machine-blindsight
!pip -q install -U "transformers>=4.43" accelerate 2>&1 | tail -n 2

## 1. Smoke test on a small base model (a few minutes)

In [ ]:
!python scripts/run_p1.py --model EleutherAI/pythia-410m --out results/pythia-410m --n-per-task 1000 --batch-size 64 --random-init-control

## 2. Main model: Qwen2.5-1.5B-Instruct

In [ ]:
!python scripts/run_p1.py --model Qwen/Qwen2.5-1.5B-Instruct --out results/qwen2.5-1.5b-instruct --n-per-task 1000 --batch-size 32 --random-init-control

## 3. Larger model: Qwen2.5-3B-Instruct (scale comparison)

In [ ]:
!python scripts/run_p1.py --model Qwen/Qwen2.5-3B-Instruct --out results/qwen2.5-3b-instruct --n-per-task 1000 --batch-size 16

## 4. Summaries

In [ ]:
import pandas as pd, glob
for f in sorted(glob.glob('results/*/summary.csv')):
    s = pd.read_csv(f)
    print('\n==', f)
    print(s[s.condition=='all'][['task','layer','report_acc','probe_acc','report_gap','n_report_wrong','blindsight_rate','blindsight_ci_lo','blindsight_ci_hi','conf_auroc','layer0_probe_acc','control_shuffled_acc']].round(3).to_string(index=False))
for f in sorted(glob.glob('results/*/steering.csv')):
    print('\n==', f)
    print(pd.read_csv(f).sort_values(['task','direction','alpha']).round(3).to_string(index=False))

In [ ]:
!cd results && zip -qr /kaggle/working/p1_results.zip . && ls -la /kaggle/working/p1_results.zip